# Fine-tuning RuModernBERT for Russian NER (factRuEval‑2016)

This notebook performs supervised fine‑tuning of the `deepvk/RuModernBERT-base` model on the factRuEval‑2016 dataset.

We use:
- token-level classification head,
- BIO labels aligned with subword tokens,
- strict span-level evaluation (seqeval),
- early stopping,
- warmup and weight decay.

This notebook establishes the main ModernBERT baseline before applying MLM pretraining, concept masking, and synthetic data augmentation.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from datasets import DatasetDict

## Tokenization and label alignment

We align BIO labels with subword tokens:
- first subword gets the label,
- remaining subwords receive -100 (ignored in loss).

In [ ]:
model_id = "deepvk/RuModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=192
    )

    labels = []
    word_ids = tokenized.word_ids()

    for i, w in enumerate(word_ids):
        if w is None:
            labels.append(-100)
        else:
            if i == 0 or w != word_ids[i - 1]:
                labels.append(example["ner_tags"][w])
            else:
                labels.append(-100)

    tokenized["labels"] = labels
    return tokenized

tokenized_ds = ds_small.map(tokenize_and_align_labels, batched=True)

## Model initialization

We load RuModernBERT with a token classification head.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

## Training configuration

We use:
- learning rate 2e‑5,
- batch size 16,
- gradient accumulation,
- early stopping,
- warmup,
- weight decay.

In [ ]:
args = TrainingArguments(
    output_dir="./modernbert_base",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    fp16=True,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50
)

## Trainer setup

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

## Model Training

In [ ]:
trainer.train()

## Predictions on test set

In [ ]:
predictions = trainer.predict(tokenized_ds["test"])

## Convert predictions to BIO labels

In [ ]:
def trainer_predictions_to_seqeval(predictions, dataset):
    logits = predictions.predictions
    preds = logits.argmax(-1)

    y_true, y_pred = [], []

    for i, example in enumerate(dataset):
        word_ids = example["word_ids"]
        labels = example["labels"]

        pred_labels = preds[i]
        out_true, out_pred = [], []

        prev_word = None
        for j, w in enumerate(word_ids):
            if w is None:
                continue
            if w != prev_word:
                out_true.append(label_names[labels[j]])
                out_pred.append(label_names[pred_labels[j]])
            prev_word = w

        y_true.append(out_true)
        y_pred.append(out_pred)

    return y_true, y_pred

y_true_base, y_pred_base = trainer_predictions_to_seqeval(predictions, tokenized_ds["test"])

## Evaluation (strict span-level F1)


In [ ]:
res_base = seqeval_strict_micro_f1(y_true_base, y_pred_base)
res_base